<a href="https://colab.research.google.com/github/DavidDimasPatty/ReelADS/blob/main/reelsADS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Download Model**


In [9]:
!pip install diffusers transformers accelerate torch torchvision imageio[ffmpeg]

**Import**

In [10]:
import torch
from diffusers.pipelines.pipeline_utils import DiffusionPipeline
import imageio
import numpy as np
import os
import cv2
from datetime import datetime
from diffusers.utils import export_to_video

**Connection**

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
model_id = "cerspense/zeroscope_v2_576w"

pipe = DiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16
)

pipe.enable_model_cpu_offload()
pipe.enable_attention_slicing()

prompt = "a cinematic drone shot of a futuristic city at sunset"

video_frames = pipe(
    prompt,
    num_frames=80,
    height=256,
    width=448,
    num_inference_steps=25
).frames


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

An error occurred while trying to fetch /root/.cache/huggingface/hub/models--cerspense--zeroscope_v2_576w/snapshots/6963642a64dbefa93663d1ecebb4ceda2d9ecb28/vae: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--cerspense--zeroscope_v2_576w/snapshots/6963642a64dbefa93663d1ecebb4ceda2d9ecb28/vae.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--cerspense--zeroscope_v2_576w/snapshots/6963642a64dbefa93663d1ecebb4ceda2d9ecb28/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
An error occurred while trying to fetch /root/.cache/huggingface/hub/models--cerspense--zeroscope_v2_576w/snapshots/6963642a64dbefa93663d1ecebb4ceda2d9ecb28/unet: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--cerspense--zeroscope_v2_576w/snapshots/6963642a64dbefa93663d1ecebb4ceda2d9ecb28/unet.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
The TextToVideoSDPipeline has been deprecated and will not receive bug fixes or feature updates after Dif

  0%|          | 0/25 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 280.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 179.81 MiB is free. Including non-PyTorch memory, this process has 14.38 GiB memory in use. Of the allocated memory 14.22 GiB is allocated by PyTorch, and 35.84 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

Save Video

In [ ]:
# fixed_frames = []

# for frame in video_frames:

#     if isinstance(frame, torch.Tensor):
#         frame = frame.detach().cpu().numpy()

#     frame = np.array(frame)

#     if frame.ndim == 3 and frame.shape[1] == 3:
#         frame = np.transpose(frame, (0, 2, 1))

#     # Normalisasi kalau 0-1
#     if frame.max() <= 1.0:
#         frame = (frame * 255).clip(0,255)

#     frame = frame.astype(np.uint8)

#     fixed_frames.append(frame)
video_np = np.array(video_frames)

# Kalau masih ada batch dimension
if video_np.ndim == 5:
    video_np = video_np[0]

print("Final video shape:", video_np.shape)

# print("Fixed shape:", fixed_frames[0].shape)

# print("Total frames:", len(fixed_frames))
# print("Shape frame 0:", fixed_frames[0].shape)
# print("Dim:", fixed_frames[0].ndim)
# print("Final frame shape:", fixed_frames[0].shape)

output_path = f"/content/drive/MyDrive/ADS_Video/generated_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.mp4"

export_to_video(video_np, output_path, fps=8)

print("Video saved:", output_path)